In [227]:
from uuid import UUID, uuid4
from datetime import datetime
from typing import TypedDict, Dict, Any, Literal, List, Optional
import json
import duckdb
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI # Use this for Gemini
from langchain_core.tools import tool
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, MessagesState, START, END
from osp_sos_analyser.dbconnector import DatabaseConnector
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
from langchain.agents import create_agent
import re

In [228]:
db = DatabaseConnector("sos_analysis.duckdb",read_only=True)
db_con = db.connect()
db_con.execute("SELECT * FROM os_logs LIMIT 10")
print(db_con.fetchall())

## sample query to fetch logs related to a specific UUID
# db_con.execute(
#     "SELECT timestamp, service, level, message FROM os_logs WHERE message LIKE ? ORDER BY timestamp",
#     ["%ba4bf98f-88ab-43a7-b68b-ef16ea47e0db%"]
# ).fetchall()

# db.fetch_all("SELECT * FROM os_logs LIMIT 10")
# print(rows)


[(None, None, 'UNKNOWN', 'unknown', 'Archived and active journals take up 799.8M in the file system.', 'system', 'system', 'sosreport-cbbhnd11fs010r02ctl02-2026-07-28-ybypdxo/sos_commands/logs/journalctl_--disk-usage', 'sosreport-cbbhnd11fs010r02ctl02-2026-07-28-ybypdxo.tar.xz', 'openstack,rhosp17,system', '17.x'), (None, None, 'UNKNOWN', 'unknown', "------------ Tue Jul 28 12:42:49 IST 2026 ------------\n[\x1b  OK  \x1b[0m] Started \x1bShow Plymouth Boot Screen\x1b.\n[\x1b  OK  \x1b[0m] Started \x1bForward Password R…s to Plymouth Directory Watch\x1b.\n[\x1b  OK  \x1b[0m] Reached target \x1bPath Units\x1b.\n[\x1b  OK  \x1b[0m] Reached target \x1bBasic System\x1b.\n[\x1b  OK  \x1b[0m] Found device \x1bMR416i-a_Gen10+ img-rootfs\x1b.\n[\x1b  OK  \x1b[0m] Reached target \x1bInitrd Root Device\x1b.\n[\x1b  OK  \x1b[0m] Finished \x1bdracut initqueue hook\x1b.\n[\x1b  OK  \x1b[0m] Reached target \x1bPreparation for Remote File Systems\x1b.\n[\x1b  OK  \x1b[0m] Reached target \x1bRemote File

In [229]:
print(db_con.execute("SELECT DISTINCT service FROM os_logs ORDER BY 1").fetchall())

[('cinder',), ('ironic',), ('neutron',), ('nova',), ('ovn',), ('podman',), ('system',), ('tripleo',)]


In [230]:
UUID_RE = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', re.I)

def extract_related_ids(log_text: str, exclude: str) -> list[str]:
    ids = set(UUID_RE.findall(log_text))
    ids.discard(exclude)
    return list(ids)

In [231]:
# db_con.execute("""
#     SELECT service, source_file, COUNT(*) 
#     FROM os_logs 
#     GROUP BY service, source_file 
#     ORDER BY service, source_file
# """).fetchall()

In [232]:
# Are there any raw archive files suggesting neutron-server ran on this host at all?
db_con.execute("""
    SELECT DISTINCT source_file FROM os_logs WHERE source_file LIKE '%neutron%'
""").fetchall()

[('sosreport-cbbhnd11fs010r02ctl02-2026-07-28-ybypdxo/var/log/containers/neutron/server.log',)]

In [233]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") or os.getenv("GROK_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY")
llm = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
).bind(parallel_tool_calls=False)

# gemini_key = os.getenv("GOOGLE_API_KEY")
# llm = init_chat_model(
#     model="gemini-3.5-flash",
#     model_provider="google_genai",
#     google_api_key=gemini_key
#     )
llm

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018F2461DB20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018F29EB3860>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'parallel_tool_calls': False}, config={}, config_factories=[])

# ---------------------------------------------------------
# 1. Define the Pydantic Schemas (The Output Structure)
# ---------------------------------------------------------

In [234]:
class InvestigationEntities(BaseModel):
    investigation_id: Optional[str] = Field(default_factory=lambda: str(uuid4()))
    resource_id: Optional[str] = Field(default=None, description="Any UUID at the center of the investigation — instance, volume, port, network, image, etc.")
    resource_type: Optional[str] = Field(default=None, description="e.g. 'instance', 'volume', 'port', 'network', 'image'")
    service: Optional[str] = Field(default=None)
    problem: Optional[str] = Field(default=None)
    cluster: Optional[str] = Field(default=None)

class TimeWindow(BaseModel):
    start: Optional[str] = Field(default=None)
    end: Optional[str] = Field(default=None)

class SearchTask(BaseModel):
    service: str
    objective: str
    query: str
    priority: int

class ExpandedQuery(BaseModel):
    summary: str
    intent: str
    entities: InvestigationEntities
    keywords: List[str]
    search_queries: List[SearchTask] = Field(default_factory=list, description="A list of search tasks to be performed, each with a service, objective, query, and priority.")
    investigation_targets: List[str]
    hypotheses: List[str] = Field(default_factory=list)
    time_window: Optional[TimeWindow] = Field(default=None)

class InvestigationState(TypedDict):
    investigation_id: UUID
    raw_query: str
    expanded_plan: Optional[ExpandedQuery]
    gathered_evidence: List[Dict[str, Any]]
    findings: Dict[str, Any]
    final_rca: Optional[str]
    next_node: str

class Evidence(BaseModel):
    service: str
    source: str
    summary: str
    raw_logs: str

# Prompt Template

In [235]:
prompt_expand = ChatPromptTemplate.from_messages([
    ("system", """You are a query enrichment agent for a Red Hat OpenStack investigation workflow.
Return JSON only with the fields: summary, intent, entities, keywords, search_queries, investigation_targets, hypotheses, time_window.

CRITICAL RULES:
- `keywords` MUST include any UUIDs, hostnames, or instance names verbatim from the incident report. Do NOT replace them with concept words like 'VM' or 'creation'.
- `keywords` should be lowercase tokens that would actually appear in OpenStack logs (e.g. 'failed', 'error', 'spawn', 'build', the UUID itself).
- `entities.service` may be: nova, cinder, neutron, glance, keystone, heat, octavia, ironic, system, or unknown.
- `entities.vm` MUST be the UUID if present in the incident.
- Use `system` for controller/compute host reboots, kernel, hardware,  podman, or OS-level symptoms.
- Use `unknown` when the incident does not explicitly identify an  OpenStack service. Do not select nova by default.
- Do not fetch more than 10 to 20 events or logs per query.
- Every selected service must be justified by words in the incident.
- Keep output lightweight and valid JSON.
"""),
    ("human", "Incident Report: {query}"),
])

In [236]:
def expand_query_node(state: InvestigationState) -> dict:
    """Your Job is to expand the raw query into a structured investigation plan for finding the Openstack Issue."""
    print("--- NODE: Expanding Query ---")
    print(f"[QUERY] raw input: {state['raw_query']}")

    structured_llm = llm.with_structured_output(ExpandedQuery, method="json_schema")

    enrichment_chain = ({"query": RunnablePassthrough()} | prompt_expand | structured_llm)

    print("[QUERY] sending prompt to LLM...")
    result = enrichment_chain.invoke(state["raw_query"])
    print(f"[QUERY] result: {result}")

    return {"expanded_plan": result}

In [237]:
def query_duckdb_logs(sql_query: str) -> str:
    """Execute SQL against DuckDB and return a readable string."""
    print(f"[DB QUERY] {sql_query}")
    try:
        rows = db_con.execute(sql_query).fetchall()
        if not rows:
            print("[DB RESULT] No matching logs.")
            return "No matching logs."
        result = "\n".join(str(row) for row in rows)
        print(f"[DB RESULT] {result}")
        return result
    except Exception as exc:
        print(f"[DB ERROR] {exc}")
        return str(exc)

def query_duckdb_logs_params(sql_query: str, params: list) -> str:
    """Execute a parameterized SQL query against DuckDB and return a readable string."""
    print(f"[DB QUERY] {sql_query}")
    print(f"[DB PARAMS] {params}")
    try:
        rows = db_con.execute(sql_query, params).fetchall()
        if not rows:
            print("[DB RESULT] No matching logs.")
            return "No matching logs."
        result = "\n".join(str(row) for row in rows)
        print(f"[DB RESULT] {result}")
        return result
    except Exception as exc:
        print(f"[DB ERROR] {exc}")
        return str(exc)


# Tool Declerations

In [238]:
@tool
def search_os_logs(
    service: str = "",
    resource_id: str = "",
    search_terms: str = "",
    limit: int = 50,
) -> str:
    """
    Search the os_logs table for Openstack service related issue. All filters you provide (service, resource_id, and every
    word in search_terms) are combined with AND — a matching row must contain ALL of them.
    Pass empty string "" for any filter you don't want to use. search_terms is
    space-separated words; keep it to 1-2 words, since every word narrows further.
    Leave service empty to search all services. Start with resource_id alone; only add
    search_terms once you have too many results to narrow down.
    you can also use os_command table if you have reasoning to troubleshoot from os level.
    limit should not be more that 10 for given query.
    - Use Table 
    """
    clauses, params = [], []
    if service:
        clauses.append("service = ?")
        params.append(service)
    if resource_id:
        clauses.append("message LIKE '%' || ? || '%'")
        params.append(resource_id)
    for term in search_terms.split():
        clauses.append("message LIKE '%' || ? || '%'")
        params.append(term)
    where = " AND ".join(clauses) if clauses else "1=1"
    sql = f"SELECT timestamp, service, level, message FROM os_logs WHERE {where} ORDER BY timestamp LIMIT {limit};"
    return query_duckdb_logs_params(sql, params)

# Manager Agent and Its State

In [239]:
from langchain.agents.middleware import SummarizationMiddleware

summarization = SummarizationMiddleware(
    model=llm,                      # reuse your Groq model; swap for a cheaper one if you want summarization itself to cost less
    trigger=("tokens", 3000),       # start summarizing once the running convo exceeds ~3k tokens
    keep=("messages", 10),          # always keep the most recent 10 messages verbatim
)

investigator_agent = create_agent(
    model=llm,
    tools=[search_os_logs],
    system_prompt="""
    You are an OpenStack and Linux Expert and RCA investigator. Given an incident, use search_os_logs iteratively to gather evidence for Openstack related issue or for UnderLine Hardware issue.

    IMPORTANT — cross-service correlation: OpenStack services rarely reference each other's
    resources by the same ID. A Nova instance log mentions volume IDs, port IDs, network IDs,
    and image IDs as *related* resources — Cinder/Neutron/Glance logs will reference those
    IDs, not the instance UUID. After your first search returns results, extract every
    UUID mentioned in the log messages (ports, volumes, networks, images, request IDs) and
    search for THOSE specifically in the services that own them (port/network IDs → neutron,
    volume/image IDs → cinder/glance) — do not just repeat the original resource_id across
    every service.

    Start broad if the responsible service is unclear. After each search, decide: is this
    evidence actually relevant to the reported problem? If not relevant or empty, try a
    different service, a different resource_id (including related IDs you've extracted),
    or different search_terms.

    Stop once you have enough relevant evidence to explain (or rule out) a root cause, or
    once you've reasonably exhausted plausible services AND plausible related IDs for this
    incident. Finish with a clear root cause analysis listing which services and which IDs
    you checked, or an explicit statement that no root cause was found along with what you tried.
    If issue described for Compute or Controller then you may also need to investigate OS level logs.
    Some time user may write cmp for compute node and ctl for controller node.
    """,
    middleware=[summarization],
)

In [240]:
from langchain_core.messages import ToolMessage, AIMessage

def investigator_node(state: InvestigationState) -> dict:
    plan = state["expanded_plan"]
    result = investigator_agent.invoke(
        {"messages": [("user", f"Incident: {plan.summary}. Known entities: {plan.entities.model_dump()}")]}
    )
    messages = result["messages"]

    # map tool_call_id -> the args that tool call was made with, so we know
    # which service/resource_id each ToolMessage's results belong to
    call_args = {}
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                call_args[tc["id"]] = tc.get("args", {})

    gathered_evidence = []
    for m in messages:
        if isinstance(m, ToolMessage):
            args = call_args.get(m.tool_call_id, {})
            gathered_evidence.append({
                "service": args.get("service") or "unknown",
                "source": m.name or "search_os_logs",
                "summary": f"resource_id={args.get('resource_id', '')} search_terms={args.get('search_terms', '')}",
                "raw_logs": m.content,
            })

    return {
        "gathered_evidence": gathered_evidence,
        "findings": {"investigator_raw": messages[-1].content},
    }




###RCA Node

In [241]:
def synthesize_rca(state: InvestigationState) -> dict:
    plan = state["expanded_plan"]
    evidence = state.get("gathered_evidence", [])
    investigator_notes = state.get("findings", {}).get("investigator_raw", "")
    evidence_text = (
        "\n\n".join(f"[{e['service']}] {e['raw_logs']}" for e in evidence)
        if evidence else "No log evidence was found for any investigated service."
    )
    rca_prompt = f"""
    Incident: {plan.summary}
    Hypotheses considered: {plan.hypotheses}
    Investigator's working notes: {investigator_notes}
    Evidence gathered:
    {evidence_text}

    Write a concise root cause analysis. If no evidence was found, say so explicitly
    and suggest what to check next (e.g. widen time window, check neutron/glance, verify UUID).
    """
    result = llm.invoke(rca_prompt)
    return {"final_rca": result.content}

### Add all nodes
workflow.add_node("QUERY_EXPAND", expand_query_node)
workflow.add_node("MANAGER", manage_agents)
workflow.add_node("NOVA_INSPECTOR", nova_inspector)
#### workflow.add_node("CINDER_INSPECTOR", cinder_inspector) # For future

1. The workflow starts by expanding the query
workflow.add_edge(START, "QUERY_EXPAND")
2. After expansion, it ALWAYS goes to the Manager
workflow.add_edge("QUERY_EXPAND", "MANAGER")
3. The Manager routes to a specialist (or ends)
workflow.add_conditional_edges(
    "MANAGER",
    route_from_manager,
    {
        "NOVA_INSPECTOR": "NOVA_INSPECTOR",
        "FINISH": END
    }
)

4. CRITICAL: Specialists ALWAYS report back to the Manager when done!
workflow.add_edge("NOVA_INSPECTOR", "MANAGER")

Compile the application
app = workflow.compile()
print("[WORKFLOW] Graph compiled successfully")


In [242]:
workflow = StateGraph(InvestigationState)
workflow.add_node("QUERY_EXPAND", expand_query_node)
workflow.add_node("INVESTIGATOR", investigator_node)
workflow.add_node("SYNTHESIZE_RCA", synthesize_rca)
workflow.add_edge(START, "QUERY_EXPAND")
workflow.add_edge("QUERY_EXPAND", "INVESTIGATOR")
workflow.add_edge("INVESTIGATOR", "SYNTHESIZE_RCA")
workflow.add_edge("SYNTHESIZE_RCA", END)
app = workflow.compile()
print("[WORKFLOW] Graph compiled successfully")


[WORKFLOW] Graph compiled successfully


# Starting The Inquery

In [243]:
from langchain_core.utils.uuid import uuid7

initial_state = {
    "raw_query": "could you tell why cbbhnd11fs010r02ctl02  rebooted",
    "expanded_plan": None,
    "gathered_evidence": [],
    "findings": {},
    "next_node": "",
}

print("🚀 Starting OpenStack Investigation Workflow...\n")

# THE FIX: Combine thread_id and recursion_limit into a single config object
run_config = {
    "configurable": {"thread_id": str(uuid7())},
    "recursion_limit": 15
}

# Pass the unified config object to the invoke method
final_state = app.invoke(initial_state, config=run_config)

print("\n" + "=" * 40)
print("🎯 INVESTIGATION COMPLETE")
print("=" * 40)

plan = final_state.get("expanded_plan")
if plan:
    print(f"\n[Parsed Intent]: {plan.intent}")
    print(f"[Extracted Time]: {plan.time_window}")

print("\n[Aggregated Findings]:")
for service, data in final_state.get("findings", {}).items():
    print(f"\n--- {service.upper()} ---")
    print(data)

print("\n[Root Cause Analysis]:")
print(final_state.get("final_rca", "No RCA generated."))

🚀 Starting OpenStack Investigation Workflow...

--- NODE: Expanding Query ---
[QUERY] raw input: could you tell why cbbhnd11fs010r02ctl02  rebooted
[QUERY] sending prompt to LLM...
[QUERY] result: summary='User asks why the controller host cbbhnd11fs010r02ctl02 rebooted.' intent='Determine the cause of the unexpected reboot of host cbbhnd11fs010r02ctl02.' entities=InvestigationEntities(investigation_id=None, resource_id=None, resource_type=None, service='system', problem=None, cluster=None) keywords=['cbbhnd11fs010r02ctl02', 'reboot', 'rebooted'] search_queries=[SearchTask(service='system', objective='Find reboot event timestamps', query='host:cbbhnd11fs010r02ctl02 AND event:reboot', priority=1), SearchTask(service='system', objective='Identify kernel panics or oops before reboot', query='host:cbbhnd11fs010r02ctl02 AND (kernel panic OR oops)', priority=2), SearchTask(service='system', objective='Check for systemd unit failures or service crashes', query='host:cbbhnd11fs010r02ctl02 AND 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01ktxkem5ze5t8584ankm5cj81` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 2356, Requested 5736. Please try again in 690ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}